In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from pandas import DataFrame
from icecream import ic

from src.utils.infer import infer_dtypes
from src.utils.paths import DATA_PATH

EXTERNAL_PATH = DATA_PATH / "external"
filename = "TB_CEP_BR_2018.csv"
filepath = EXTERNAL_PATH / "raw" / filename

if not filepath.exists():
    raise FileNotFoundError

In [2]:
df = pd.read_csv(
    filepath,
    delimiter=";",
    dtype=str,
    encoding="utf_8",
)
df.shape

(1582107, 6)

In [3]:
# scrap
# idx_headers = df[df[0].str.contains("CEP")].index.tolist()

# # table a
# df_a_headers: list[str] = df.iloc[idx_headers[0]].tolist()

# df_a = df.iloc[idx_headers[0] + 1 : idx_headers[1]].copy().reset_index(drop=True)
# df_a.columns = df_a_headers

# # table b
# df_b_headers: list[str] = df.iloc[idx_headers[1]].tolist()
# df_b_headers[0] = 'CEP'
# df_b_headers.pop(-1)

# df_b = df.iloc[idx_headers[1]+1:].copy().reset_index(drop=True)
# # df_b.columns = df_b_headers

# df_b

# # df_a first table, df_b second table. indexes reset

In [4]:
# this file has two tables stacked one on top of the other
# define table A as the table that appears with columns:
#   CEC, UF, CIDADE, BAIRRO, LOGRADOURO, COMPLEMENTO
# define table B as the table that appears second with columns:
#   CEC, UF, CIDADE, BAIRRO, LOGRADOURO

# tables A and B have an intersection
# prioritise table A
#   natural to assume priority since first
#   uses the additional column
# any CEP not in A but in B, include with A
# have a separate table B

# want to make
#   single reference table: (on CEP)
#       A u B\A (prefer A, supplement with unique B)
#   extra table
#       B (whole of B)

# partition
idx_repeat_header = df[df["CEP"].str.contains("CEP")].index[0]
df_A = df.iloc[:idx_repeat_header, :].copy()
df_B = df.iloc[idx_repeat_header + 1 :, :-1].copy().reset_index(drop=True)

df.shape[0] == df_A.shape[0] + df_B.shape[0] + 1
# the +1 is the row of the extra headers

True

In [5]:
# exclusive or (xor) operator ^
# distinct elements in A, union, distinct elements in B
# symmetric difference
# A\B u B\a
symm_diff = set(df_A["CEP"]) ^ set(df_B["CEP"])

# on col CEP, A\B
cep_AmB = set(df_A["CEP"]) - set(df_B["CEP"])

# on col CEP B\A
cep_BmA = set(df_B["CEP"]) - set(df_A["CEP"])

In [6]:
# unique CEPs on table B
cep_BmA_list = list(sorted(cep_BmA))
df_BmA = df[df["CEP"].isin(cep_BmA_list)]
# infer_dtypes(df_BmA)

# AuBmA means A u (B\A) means A union (B set minus A)
df_AuBmA = pd.concat([df_A, df_BmA])

In [7]:
# then I have two tables: df_AuBmA unique CEP, and df_B unique CEP

# TB_CEP_BR_2018__AuBmA.csv
df_AuBmA_attrs = infer_dtypes(df_AuBmA)

# TB_CEP_BR_2018__B.csv
df_B_attrs = infer_dtypes(df_B)

CEP
UF
CIDADE
BAIRRO
LOGRADOURO
COMPLEMENTO
CEP
UF
CIDADE
BAIRRO
LOGRADOURO


In [8]:
# finding sql server data types:
pd.set_option("display.max_columns", None)

# df_AuBmA_attrs
"""
CEP
    unique
    no nulls
    max 8
    fixed length
UF
    no nulls
    max 14
CIDADE
    max 62
    nulls
    nonascii
BAIRRO
    max 65
    nulls
    nonascii
LOGRADOURO
    max 132
    nulls
    nonascii
COMPLEMENTO
    max 78
    nulls
    nonascii
"""

# df_B_attrs
"""
no nulls
CEP
    max 8
    unique
    no nulls
    fixed lengths
UF
    max 2
    no nulls
    fixed lengths
CIDADE
    max 23
    no null
    nonascii
BAIRRO
    max 63
    no nulls
    nonascii
LOGRADOURO
    max 71
    no nulls
    nonascii
"""
print()

In [9]:
# df without repeat header
# mrh means minus repeat headers
df_mrh = df[~df["CEP"].str.contains("CEP")]
df_mrh_attrs = infer_dtypes(df_mrh)

CEP
UF
CIDADE
BAIRRO
LOGRADOURO
COMPLEMENTO


In [10]:
"""
no nulls: cep, uf, cidade

CEP char(8) not null
UF varchar(20) not null
    max 14
CIDADE nvarchar(70) not null
    max 62
    nonascii
BAIRRO nvarchar(70)
    max 65
    nonascii
LOGRADOURO nvarchar(150)
    max 132
    nonascii
COMPLEMENTO nvarchar(100)
    max 78
    nonascii
"""
df_mrh_attrs

,has_unique_entries,has_nulls,where_nulls,total_nulls,sorted_chars_used,total_unique_chars_used,has_ascii,has_non_ascii,has_prefix_zero,dice_sim_to_ascii,dice_sim_to_non_ascii,min_str_value,max_str_value,entry_lengths,total_unique_entry_lengths,max_entry_length,is_fixed_length,chars_used_subset_of_numeric,has_prefix_dash,has_digits,has_hex_digits,has_decimal,dice_sim_to_digits,dice_sim_to_hex_digits,min_numeric_value,max_numeric_value,has_dash,has_colon,has_space,has_exactly_two_entries
CEP,0,0,[],0,0123456789,10,1,0,1,0.181818,0.0,01001000,99980974,[8],1,8,1,1,0,1,1,0,1.0,0.625,1001000,99980974,0,0,0,0
UF,0,0,[],0,-ABCDEFGIJLMNOPRSTadiorstv,27,1,0,0,0.425197,0.0,AC,TO - Povoado,"[2, 13, 14]",3,14,0,0,0,0,1,0,0.0,0.326531,<NA>,<NA>,1,0,1,0
CIDADE,0,0,[],0,'()-01234569ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefg...,82,1,1,0,0.714286,0.343434,Abacate da Pedreira (Macapá),Óleo,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",52,62,0,0,0,1,1,0,0.173913,0.384615,<NA>,<NA>,1,0,1,0
BAIRRO,0,1,"[36042, 36754, 36976, 36977, 37167, 38282, 384...",12018,"""'()+,-./0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ...",99,1,1,0,0.733668,0.416,14 de Novembro,Índios,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",63,65,0,0,0,1,1,1,0.183486,0.363636,<NA>,<NA>,1,0,1,0
LOGRADOURO,0,1,"[58128, 62746, 67081, 67088, 67099, 67100, 757...",10040,"!'()*,-./0123456789:;=ABCDEFGHIJKLMNOPQRSTUVW...",119,1,1,0,0.721461,0.503145,1 DF-130,Área Área Especial 6/8,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",112,132,0,0,0,1,1,1,0.155039,0.312057,9.0,28.0,1,1,1,0
COMPLEMENTO,0,1,"[0, 1, 2, 6, 7, 8, 9, 14, 15, 16, 23, 24, 30, ...",1555797,"!'(),-./0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ`...",100,1,1,0,0.72,0.4375,10º. Batalhão de Polícia,ÚNICA LOGISTIC,"[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...",77,78,0,0,0,1,1,1,0.181818,0.360656,<NA>,<NA>,1,0,1,0


In [11]:
print(df_A.shape[0])
print(df_B.shape[0])
df_A

732763
849343


,CEP,UF,CIDADE,BAIRRO,LOGRADOURO,COMPLEMENTO
0,01001000,SP,São Paulo,Sé,Praça da Sé - lado ímpar,NaN
1,01001001,SP,São Paulo,Sé,Praça da Sé - lado par,NaN
2,01001010,SP,São Paulo,Sé,Rua Filipe de Oliveira,NaN
3,01001900,SP,São Paulo,Sé,"Praça da Sé, 108",UNESP - Universidade Estadual Júlio de Mesqui...
4,01001901,SP,São Paulo,Sé,"Praça da Sé, 371",Edifício Santa Lídia
...,...,...,...,...,...,...
732758,99975970,RS - Distrito,São João Bosco (Ciríaco),Centro,"Rua Principal, s/n",AGC São João Bosco
732759,99978000,RS - Distrito,Cruzaltinha (Ciríaco),NaN,NaN,NaN
732760,99980000,RS,David Canabarro,NaN,NaN,NaN
732761,99980970,RS,David Canabarro,Centro,"Rua Adelino Gazzoni, 160",AC David Canabarro


In [12]:
df_B

,CEP,UF,CIDADE,BAIRRO,LOGRADOURO
0,01001000,SP,São Paulo,Sé,Praça da Sé
1,01001001,SP,São Paulo,Sé,Praça da Sé
2,01001010,SP,São Paulo,Sé,Rua Filipe de Oliveira
3,01002000,SP,São Paulo,Sé,Rua Direita
4,01002001,SP,São Paulo,Sé,Rua Direita
...,...,...,...,...,...
849338,99074530,RS,Passo Fundo,Lucas Araújo,Rua Padre Champagnt
849339,99074540,RS,Passo Fundo,Lucas Araújo,Rua Pio X
849340,99074550,RS,Passo Fundo,Lucas Araújo,Rua Guia Lopes
849341,99074570,RS,Passo Fundo,Lucas Araújo,Rua Nossa Senhora dos Passos


In [13]:
repeat_cep = df_mrh[df_mrh["CEP"].duplicated()]["CEP"].tolist()
len(repeat_cep)

587016

In [14]:
start = 300000
diff = 50

end = 10

# idxs = repeat_cep[start:end]
idxs = repeat_cep[start : start + diff]
for i in idxs:
    print(df_mrh[df_mrh["CEP"] == i])

              CEP  UF    CIDADE  BAIRRO         LOGRADOURO COMPLEMENTO
353373   41290220  BA  Salvador  Pirajá  Travessa Ferreira         NaN
1167545  41290220  BA  Salvador  Pirajá  Travessa Ferreira         NaN
              CEP  UF    CIDADE  BAIRRO            LOGRADOURO COMPLEMENTO
353374   41290221  BA  Salvador  Pirajá  1ª Travessa Ferreira         NaN
1167546  41290221  BA  Salvador  Pirajá  1ª Travessa Ferreira         NaN
              CEP  UF    CIDADE  BAIRRO            LOGRADOURO COMPLEMENTO
353375   41290222  BA  Salvador  Pirajá  2ª Travessa Ferreira         NaN
1167547  41290222  BA  Salvador  Pirajá  2ª Travessa Ferreira         NaN
              CEP  UF    CIDADE  BAIRRO            LOGRADOURO COMPLEMENTO
353376   41290223  BA  Salvador  Pirajá  3ª Travessa Ferreira         NaN
1167548  41290223  BA  Salvador  Pirajá  3ª Travessa Ferreira         NaN
              CEP  UF    CIDADE  BAIRRO            LOGRADOURO COMPLEMENTO
353377   41290224  BA  Salvador  Pirajá  4ª Tra